### Scrape data from NBARapm.com

In [5]:
import pandas as pd
import numpy as np
import time
import unicodedata
import re
import os
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

# Load player list
players = pd.read_csv("tankathon_players.csv")
player_col = "name"

# Progress file with timestamp to avoid conflicts
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
PROGRESS_FILE = f"peak_darko_results_{TIMESTAMP}.csv"
CHECKPOINT_INTERVAL = 5  # Save after every N players

# Set up Selenium (headless Chrome)
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=options)

def normalize_name(name, variant=0):
    """
    Normalize player names for nbarapm URL format with multiple variants:
    - variant 0: Remove accents, remove punctuation, normalize suffixes
    - variant 1: Keep periods in initials (J.R. → j.r.)
    - variant 2: Remove periods but keep as initials (J.R. → jr)
    - variant 3: Lowercase everything, remove all special chars
    """
    # Remove accents
    name = ''.join(
        c for c in unicodedata.normalize('NFD', name)
        if unicodedata.category(c) != 'Mn'
    )
    
    if variant == 0:
        # Default: remove all punctuation
        name = re.sub(r"[.\-']", '', name)
        name = re.sub(r"\b(JR|Jr|jr)\b", "Jr", name)
        name = re.sub(r"\bIII\b", "III", name)
        name = re.sub(r"\bII\b", "II", name)
        name = re.sub(r"\bIV\b", "IV", name)
    elif variant == 1:
        # Keep periods, lowercase initials
        name = re.sub(r"\b([A-Z])\.([A-Z])\.", lambda m: f"{m.group(1).lower()}.{m.group(2).lower()}.", name)
        name = re.sub(r"[-']", '', name)
        name = re.sub(r"\b(JR|Jr|jr)\b", "Jr", name)
    elif variant == 2:
        # Remove periods, lowercase initials
        name = re.sub(r"\b([A-Z])\.([A-Z])\.", lambda m: f"{m.group(1).lower()}{m.group(2).lower()}", name)
        name = re.sub(r"[.\-']", '', name)
        name = re.sub(r"\b(JR|Jr|jr)\b", "Jr", name)
    elif variant == 3:
        # Aggressive: lowercase everything, remove all special chars
        name = name.lower()
        name = re.sub(r"[.\-']", '', name)
        name = re.sub(r"\b(jr|iii|ii|iv)\b", lambda m: m.group(1), name)
    
    # Clean up whitespace
    name = re.sub(r"\s+", " ", name.strip())
    return name.replace(" ", "_")

def check_page_validity(soup, player_name):
    """Check if the page loaded successfully and contains player data"""
    # Check for explicit 404 or error messages
    page_text = soup.get_text().lower()
    if "404" in page_text or "not found" in page_text or "page not found" in page_text:
        return False
    
    # More lenient check - just see if there's ANY content that looks like data
    # Check for player name or any tabulator/table content
    if soup.find("div", class_=re.compile(r"tabulator")) or \
       soup.find("table") or \
       soup.find(string=re.compile(r"Peak DARKO", re.IGNORECASE)) or \
       soup.find(string=re.compile(r"DARKO", re.IGNORECASE)):
        return True
    
    # If the page has substantial content (not just a blank page), consider it valid
    if len(page_text) > 500:  # Arbitrary threshold for "substantial content"
        return True
    
    return False

def get_peak_darko(player_name, max_variants=4):
    """
    Try multiple name normalization variants until one works
    """
    attempted_urls = []
    
    for variant in range(max_variants):
        url_name = normalize_name(player_name, variant=variant)
        url = f"https://www.nbarapm.com/player/{url_name}"
        
        # Skip if we already tried this exact URL
        if url in attempted_urls:
            continue
        
        attempted_urls.append(url)
        
        try:
            driver.get(url)
            time.sleep(2.5)  # wait for JS to load
            soup = BeautifulSoup(driver.page_source, "html.parser")
            
            # Check if page is valid (but don't skip variants if validation is uncertain)
            if not check_page_validity(soup, player_name):
                if variant < max_variants - 1:
                    print(f"  ↻ Variant {variant} failed validation for {player_name}, trying next... ({url_name})")
                    continue
            
            # Find the "Peak DARKO" label div - try multiple approaches
            label_div = soup.find("div", string=lambda t: t and "Peak DARKO" in t)
            
            # Alternative: search more broadly
            if not label_div:
                label_div = soup.find(string=re.compile(r"Peak DARKO", re.IGNORECASE))
                if label_div:
                    label_div = label_div.find_parent("div")
            
            if not label_div:
                if variant < max_variants - 1:
                    print(f"  ↻ Peak DARKO not found (variant {variant}), trying next... ({url_name})")
                    continue
                else:
                    print(f"⚠️ {player_name}: Peak DARKO row not found in any variant. Last URL: {url}")
                    return None
            
            # Get its parent row (contains all metrics)
            parent_row = label_div.find_parent("div", class_="tabulator-row")
            
            # Alternative: if no tabulator-row class, try finding any parent div that contains a strong tag
            if not parent_row:
                parent_row = label_div.find_parent("div")
                # Search within broader context
                for _ in range(3):  # Try up to 3 parent levels
                    if parent_row and parent_row.find("strong"):
                        break
                    if parent_row:
                        parent_row = parent_row.find_parent("div")
            
            if not parent_row:
                if variant < max_variants - 1:
                    print(f"  ↻ Parent row not found (variant {variant}), trying next... ({url_name})")
                    continue
                else:
                    print(f"⚠️ {player_name}: Parent row not found in any variant. Last URL: {url}")
                    return None
            
            # Within that row, find the <strong> tag — that holds the number
            strong_tag = parent_row.find("strong")
            if not strong_tag:
                if variant < max_variants - 1:
                    print(f"  ↻ Value not found (variant {variant}), trying next... ({url_name})")
                    continue
                else:
                    print(f"⚠️ {player_name}: Peak DARKO value not found in any variant. Last URL: {url}")
                    return None
            
            value = strong_tag.get_text(strip=True)
            try:
                result = float(value)
                if variant > 0:
                    print(f"✓ {player_name}: {result} (found with variant {variant}: {url_name})")
                else:
                    print(f"{player_name}: {result}")
                return result
            except ValueError:
                if variant < max_variants - 1:
                    print(f"  ↻ Could not parse value '{value}' (variant {variant}), trying next...")
                    continue
                else:
                    print(f"⚠️ {player_name}: Could not parse value '{value}' in any variant")
                    return None
                    
        except Exception as e:
            if variant < max_variants - 1:
                print(f"  ↻ Error with variant {variant} for {player_name}, trying next... ({str(e)[:50]})")
                continue
            else:
                print(f"❌ Error fetching {player_name} (all variants failed): {e}")
                return None
    
    return None

def find_latest_progress_file():
    """Find the most recent progress file to resume from"""
    pattern = re.compile(r'peak_darko_results_(\d{8}_\d{6})\.csv')
    files = [f for f in os.listdir('.') if pattern.match(f)]
    
    if not files:
        return None
    
    # Sort by timestamp (most recent first)
    files.sort(reverse=True)
    return files[0]

def load_progress():
    """Load existing progress if available"""
    # First check if user wants to resume from an existing file
    existing_file = find_latest_progress_file()
    
    if existing_file:
        response = input(f"\n📂 Found existing progress file: {existing_file}\n   Resume from this file? (y/n): ").lower()
        if response == 'y':
            try:
                df = pd.read_csv(existing_file)
                print(f"✓ Loaded {len(df)} players from {existing_file}")
                # Update global PROGRESS_FILE to continue using this file
                global PROGRESS_FILE
                PROGRESS_FILE = existing_file
                return df
            except Exception as e:
                print(f"⚠️  Could not load file: {e}")
                print("   Starting fresh...")
    
    return pd.DataFrame(columns=["Player", "Peak DARKO"])

def save_progress(df):
    """Save current progress to CSV with retry logic"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            df.to_csv(PROGRESS_FILE, index=False)
            return True
        except PermissionError:
            if attempt < max_retries - 1:
                print(f"⚠️  File is locked. Retrying in 2 seconds... (attempt {attempt + 1}/{max_retries})")
                time.sleep(2)
            else:
                # Try alternative filename
                alt_filename = f"peak_darko_results_backup_{datetime.now().strftime('%H%M%S')}.csv"
                print(f"⚠️  Could not save to {PROGRESS_FILE}. Trying {alt_filename}...")
                try:
                    df.to_csv(alt_filename, index=False)
                    print(f"✓ Saved to {alt_filename} instead")
                    return True
                except Exception as e:
                    print(f"❌ Could not save progress: {e}")
                    return False
        except Exception as e:
            print(f"❌ Error saving progress: {e}")
            return False

# Load existing progress
results_df = load_progress()
scraped_players = set(results_df['Player'].tolist())

print(f"\n🚀 Starting scrape. Will save to: {PROGRESS_FILE}")
print(f"📊 {len(scraped_players)} players already completed.")
print(f"📊 {len(players) - len(scraped_players)} players remaining.\n")

# Scrape players
results = results_df.to_dict('records')
count = 0

try:
    for i, row in players.iterrows():
        name = row[player_col]
        
        # Skip if already scraped
        if name in scraped_players:
            print(f"⏭️  Skipping {name} (already scraped)")
            continue
        
        peak_darko = get_peak_darko(name)
        results.append({"Player": name, "Peak DARKO": peak_darko})
        count += 1
        
        # Save progress at regular intervals
        if count % CHECKPOINT_INTERVAL == 0:
            results_df = pd.DataFrame(results)
            if save_progress(results_df):
                print(f"💾 Progress saved ({len(results)} total players)\n")
        
        # Optional: add a small delay between requests to be polite
        time.sleep(0.5)

except KeyboardInterrupt:
    print("\n\n⚠️  Script interrupted by user. Saving progress...")
except Exception as e:
    print(f"\n\n❌ Unexpected error: {e}. Saving progress...")
finally:
    # Always save final results, even if interrupted
    driver.quit()
    results_df = pd.DataFrame(results)
    save_progress(results_df)
    
    # Print summary
    successful = results_df['Peak DARKO'].notna().sum()
    failed = results_df['Peak DARKO'].isna().sum()
    print(f"\n✅ Final results saved to {PROGRESS_FILE}")
    print(f"📊 Successfully scraped: {successful}/{len(results_df)} players")
    if failed > 0:
        print(f"⚠️  Failed: {failed} players")
        print("\nFailed players:")
        print(results_df[results_df['Peak DARKO'].isna()]['Player'].tolist())


🚀 Starting scrape. Will save to: peak_darko_results_20251027_184234.csv
📊 0 players already completed.
📊 1434 players remaining.

Justin Reed: -1.5
Peter John Ramos: -0.2
Tony Allen: 2.6
  ↻ Parent row not found (variant 0), trying next... (Ha_SeungJin)
⚠️ Ha Seung-Jin: Parent row not found in any variant. Last URL: https://www.nbarapm.com/player/ha_seungjin
Matt Freije: -2.9
💾 Progress saved (5 total players)

Devin Harris: 2.6
Jameer Nelson: 3.3
Kris Humphries: -1.3
  ↻ Parent row not found (variant 0), trying next... (Rashad_Wright)
⚠️ Rashad Wright: Parent row not found in any variant. Last URL: https://www.nbarapm.com/player/rashad_wright
  ↻ Parent row not found (variant 0), trying next... (Marcus_Douthit)
⚠️ Marcus Douthit: Parent row not found in any variant. Last URL: https://www.nbarapm.com/player/marcus_douthit
💾 Progress saved (10 total players)

Luis Flores: -2.4
David Harrison: -0.9
Antonio Burks: -1.7
Andre Iguodala: 3.6
Shaun Livingston: 0.6
💾 Progress saved (15 total 

### Merge Datasets

In [7]:
# Load both CSVs
tankathon = pd.read_csv("tankathon_players.csv")
darko = pd.read_csv("peak_darko_results_final.csv")

# Check column names
print("Tankathon columns:", tankathon.columns.tolist())
print("DARKO columns:", darko.columns.tolist())

# Make sure names line up — e.g. 'name' vs 'Player'
# Rename if necessary
darko = darko.rename(columns={"Player": "name"})
darko = pd.DataFrame(darko)

# Merge the two datasets on player name
merged = pd.merge(tankathon, darko, on="name", how="left")

# Check the result
df = pd.DataFrame(merged)

# Extract year (4-digit number) from 'title' column
df["year"] = df["title"].str.extract(r"(\d{4})")

# Drop title
df = df.drop(columns=["title", "url"])
df.head

Tankathon columns: ['3/4_sprint', '3p%', '3pa_rate', '3pm-3pa', 'ast', 'ast/to', 'ast/usg', 'birth_date', 'blk', 'bpm', 'dbpm', 'draft_age', 'draft_position', 'draft_year', 'drtg', 'dws/40', 'effective_fg%', 'fg%', 'fgm-fga', 'ft%', 'fta_rate', 'ftm-fta', 'g', 'height', 'high_school', 'home_town', 'lane_agility', 'max_vertical', 'mp', 'name', 'nation', 'obpm', 'ortg', 'ows/40', 'per', 'pf', 'position', 'pre-draft_team', 'proj_nba_3p%', 'pts', 'reb', 'school_year', 'shuttle', 'standing_reach', 'stl', 'title', 'to', 'true_shooting_%', 'url', 'usg%', 'weight', 'wingspan', 'ws/40']
DARKO columns: ['Player', 'Peak DARKO']


<bound method NDFrame.head of       3/4_sprint    3p%  3pa_rate  3pm-3pa  ast  ast/to  ast/usg  \
0            NaN  0.237     0.142  0.5-2.3  1.6    0.49      NaN   
1           3.60    NaN       NaN      NaN  NaN     NaN      NaN   
2           3.19  0.297     0.183  0.7-2.4  3.5    1.11      NaN   
3            NaN    NaN       NaN      NaN  NaN     NaN      NaN   
4           3.25  0.356     0.390  2.3-6.4  1.1    0.88     0.23   
...          ...    ...       ...      ...  ...     ...      ...   
1429        3.30  0.385     0.272  1.6-4.3  4.9    2.00     0.87   
1430         NaN  0.382     0.249  1.4-3.6  2.0    1.32     0.44   
1431        3.12  0.395     0.551  3.7-9.4  1.2    0.95     0.27   
1432        3.12  0.346     0.308  1.7-4.9  1.4    0.62     0.30   
1433        3.50  0.250     0.084  0.2-0.7  0.9    0.67     0.27   

        birth_date  blk   bpm  ...  standing_reach  stl   to  true_shooting_%  \
0     Jan 16, 1982  0.9   NaN  ...             NaN  0.7  3.2            

### Clean dataset

In [11]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Auto-detect width
pd.set_option('display.max_colwidth', None) # Show full column content

# Reorder columns
ordered_columns = [
    # Identifiers
    'name', 'Player', 'draft_year', 'draft_position',
    # Demographics
    'birth_date', 'draft_age', 'height', 'weight', 'wingspan', 
    # Basic counting stats
    'pts', 'reb', 'ast', 'stl', 'blk', 'pf', 'to', 'g', 'mp',
    # Athleticism
    'max_vertical', 'lane_agility', 'shuttle', '3/4_sprint', 'standing_reach',
    # College / background
    'pre-draft_team', 'school_year', 'high_school', 'home_town', 'nation',
    #
    # Shooting & scoring efficiency
    'fgm-fga', 'fg%', '3pm-3pa', '3p%', '3pa_rate', 'fta_rate', 'ftm-fta', 'ft%', 'effective_fg%', 'true_shooting_%',
    # Play style / advanced offense
    'usg%', 'ast/usg', 'ast/to',
    # Value metrics
    'per', 'ortg', 'drtg', 'obpm', 'dbpm', 'bpm', 'ows/40', 'dws/40', 'ws/40', 'proj_nba_3p%',
    # Outcome
    'Peak DARKO'
]


# Clean draft position
def clean_draft_position(pos):
    if isinstance(pos, str):
        # Extract digits (e.g., "1st" → "1")
        match = re.search(r"\d+", pos)
        if match:
            return int(match.group())
        # Handle undrafted or missing
        if "undrafted" in pos.lower() or "n/a" in pos.lower():
            return 61
    # If it's already numeric or NaN
    return pos if pd.notna(pos) else 61

# Apply cleaning
df["draft_position"] = df["draft_position"].apply(clean_draft_position)

# Convert to numeric
df["draft_position"] = pd.to_numeric(df["draft_position"], errors="coerce").fillna(61).astype(int)

# Reorder
df = df[[c for c in ordered_columns if c in df.columns]]

# Sort values
df = df.sort_values(by=["draft_year", "draft_position"], ascending=[False, True])

def height_to_inches(height_str):
    if pd.isna(height_str):
        return None
    
    # Get everything before the first double quote
    height_part = str(height_str).split('"')[0].strip()
    
    # Split by apostrophe to get feet and inches
    parts = height_part.split("'")
    if len(parts) == 2:
        feet = int(parts[0])
        inches = float(parts[1])
        return feet * 12 + inches
    return None

df['height_in'] = df['height'].apply(height_to_inches)

df['height'] = df['height'].str.split('(').str[0].str.strip()

df['weight_lbs'] = df['weight'].str.split(' ').str[0].str.strip()

# Check results
# Save the cleaned dataset
df.to_csv("final_merged_dataset.csv", index=False)
print(f"✅ Cleaned dataset saved! Shape: {df.shape}")

✅ Cleaned dataset saved! Shape: (1434, 53)
